# C-MNIST border FAST/SLOW-only generator

Generates `fast_r10` and `slow_g45` only. Run the first code cell to generate the dataset. The second cell recomputes statistics and checks the intended order flip.

In [ ]:
import os
import csv
import json
import random
from pathlib import Path

import cv2
import numpy as np
import torch
from PIL import Image
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import torchvision
from torchvision import transforms


# ============================================================
# C-MNIST border-calibrated FAST/SLOW-only generator
# ============================================================
#
# Purpose
# -------
# Generate a two-condition C-MNIST dataset:
#   fast_r10 : digit stroke differs from gray mean only in R by +10% of RGB range.
#   slow_g45 : digit stroke differs from gray mean only in G by +45% of RGB range.
#
# Raw/human intuition:
#   slow_g45 has much larger visible/raw RGB difference than fast_r10.
#
# Dataset-statistics trick:
#   The 3px border is a label-independent calibration region.
#   It has almost no R variance but very large G variance.
#   Therefore empirical normalization can make the small R feature effectively larger
#   than the large G feature:
#       (26/255) / sigma_R > (115/255) / sigma_G.
#
# Important
# ---------
# The border must be independent of label and condition semantics. It should not carry
# class information. It only controls empirical channel std.
#
# Output structure
# ----------------
# output_dir/
#   colors.json
#   mean.json
#   stats.json
#   train/images/fast_r10/000000.png
#   train/images/slow_g45/000000.png
#   train/manifest.csv
#   train/manifest_long.csv
#   test/...
#
# Training note
# -------------
# This generator writes a two-condition manifest. If your loader hard-codes
# NUM_CONDITIONS=4, set it to 2 or make it infer the number of cond{i}_ columns.
# Use p_values: [10, 45].
#


def set_seed(seed: int = 0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def tensor_to_pil_image(x: torch.Tensor) -> Image.Image:
    x = x.detach().cpu().clamp(0, 1)
    arr = (x.permute(1, 2, 0).numpy() * 255.0).round().astype(np.uint8)
    if arr.shape[2] == 1:
        arr = arr[:, :, 0]
    return Image.fromarray(arr)


def clip_uint8_value(x):
    return int(np.clip(round(float(x)), 0, 255))


def rgb_uint8_to_float(rgb):
    return np.asarray([clip_uint8_value(v) for v in rgb], dtype=np.float64) / 255.0


def rgb_hex_from_uint8(rgb):
    r, g, b = [clip_uint8_value(v) for v in rgb]
    return f'#{r:02X}{g:02X}{b:02X}'


def get_max_bounding_box(mask):
    y_indices, x_indices = np.where(mask)
    if len(y_indices) == 0:
        return (0, 0, 0, 0)
    return (np.min(y_indices), np.min(x_indices), np.max(y_indices), np.max(x_indices))


def get_ratio(mask):
    bbox = get_max_bounding_box(mask)
    min_y, min_x, max_y, max_x = bbox
    bbox_area = (max_y - min_y + 1) * (max_x - min_x + 1)
    if bbox_area < 10:
        return 0
    mask_area = torch.sum(mask)
    return (mask_area / bbox_area).item()


def get_exact_digit_mask(image, target_pixels, image_size, bias):
    """Return a binary digit mask with exactly target_pixels active pixels."""
    binary = image.squeeze() > bias
    pixel_count = int(binary.sum().item())
    if pixel_count == 0:
        return torch.zeros(image_size, image_size, dtype=torch.float32)

    min_y, min_x, max_y, max_x = get_max_bounding_box(binary)
    actual_data = image[:, min_y:max_y + 1, min_x:max_x + 1]

    scale = (target_pixels / pixel_count) ** 0.5
    new_h = max(1, min(image_size, round(scale * actual_data.shape[1])))
    new_w = max(1, min(image_size, round(scale * actual_data.shape[2])))

    np_img = torch.permute(actual_data, [1, 2, 0]).numpy()
    resized = cv2.resize(src=np_img, dsize=(new_w, new_h), interpolation=cv2.INTER_LINEAR)
    resized = torch.as_tensor(resized, dtype=torch.float32)
    if resized.ndim == 3:
        resized = resized.squeeze(-1)

    canvas = torch.zeros(image_size, image_size, dtype=torch.float32)
    x_off = max(0, (image_size - new_w) // 2)
    y_off = max(0, (image_size - new_h) // 2)
    canvas[y_off:y_off + resized.shape[0], x_off:x_off + resized.shape[1]] = resized

    flat = canvas.flatten().clone()
    top_idx = torch.topk(flat, k=target_pixels, largest=True).indices
    out = torch.zeros(image_size * image_size, dtype=torch.float32)
    out[top_idx] = 1.0
    return out.reshape(image_size, image_size)


def load_filtered_mnist(mnist_dir, min_mask_ratio, bias):
    mnist_train = torchvision.datasets.MNIST(
        root=mnist_dir,
        train=True,
        download=False,
        transform=transforms.Compose([transforms.ToTensor()]),
    )
    mnist_test = torchvision.datasets.MNIST(
        root=mnist_dir,
        train=False,
        download=False,
        transform=transforms.Compose([transforms.ToTensor()]),
    )

    source_records = [[], []]
    filtered_img_ids = [[], []]

    print('calculate mask ratio...')
    for split_idx, dataset in enumerate([mnist_train, mnist_test]):
        for index, (x, label) in tqdm(enumerate(dataset), total=len(dataset)):
            label = int(label)
            ratio = get_ratio(x.squeeze() > bias)
            if ratio < min_mask_ratio:
                filtered_img_ids[split_idx].append(index)
            else:
                source_records[split_idx].append({
                    'source_pool_index': len(source_records[split_idx]),
                    'source_mnist_index': int(index),
                    'label': label,
                    'image': x,
                })
        print(f"split={split_idx}: kept {len(source_records[split_idx])}, filtered {len(filtered_img_ids[split_idx])}")
    return source_records, filtered_img_ids


def make_condition_source_permutations(num_sources, condition_ids_order, seed_value, split_idx,
                                       shuffle_source_pool_per_condition=True):
    """Create one deterministic permutation per condition.

    Every condition receives the exact same source MNIST multiset and label histogram.
    Only the row-wise pairing is decorrelated by independent shuffling.
    """
    if num_sources <= 0:
        raise ValueError('The source pool is empty after filtering. Relax min_mask_ratio or check MNIST loading.')

    base = np.arange(num_sources, dtype=np.int64)
    permutations = {}
    seen = set()

    for cond_idx, cid in enumerate(condition_ids_order):
        perm = base.copy()
        rng_seed = int(seed_value) + 1_000_003 * (int(split_idx) + 1) + 9_176 * (int(cond_idx) + 1)
        rng = np.random.default_rng(rng_seed)
        if shuffle_source_pool_per_condition:
            rng.shuffle(perm)
        permutations[cid] = perm.tolist()

        key = tuple(permutations[cid])
        if key in seen and num_sources > 1:
            shift = (cond_idx + 1) % num_sources
            permutations[cid] = np.roll(np.asarray(permutations[cid], dtype=np.int64), shift).tolist()
            key = tuple(permutations[cid])
        seen.add(key)

    expected = list(range(num_sources))
    for cid, perm in permutations.items():
        assert sorted(perm) == expected, f'{cid} does not contain the exact source pool.'
    return permutations


def label_histogram_from_records(records):
    counts = [0] * 10
    for rec in records:
        counts[int(rec['label'])] += 1
    return counts


def label_histogram_from_permutation(source_pool, perm):
    counts = [0] * 10
    for source_pos in perm:
        counts[int(source_pool[int(source_pos)]['label'])] += 1
    return counts


def compute_empirical_stats(image_paths):
    sum_rgb = np.zeros(3, dtype=np.float64)
    sum_sq_rgb = np.zeros(3, dtype=np.float64)
    num_pixels = 0
    for path in tqdm(image_paths, desc='empirical stats'):
        arr = np.asarray(Image.open(path).convert('RGB'), dtype=np.float64) / 255.0
        flat = arr.reshape(-1, 3)
        sum_rgb += flat.sum(axis=0)
        sum_sq_rgb += (flat ** 2).sum(axis=0)
        num_pixels += flat.shape[0]
    mean = sum_rgb / num_pixels
    var = np.maximum(sum_sq_rgb / num_pixels - mean ** 2, 0.0)
    std = np.sqrt(var)
    return mean, std


def deterministic_rng(seed, *keys):
    """Deterministic numpy RNG keyed by split/sample/condition."""
    val = int(seed)
    for k in keys:
        if isinstance(k, str):
            for ch in k.encode('utf-8'):
                val = (val * 1315423911 + ch) & 0xFFFFFFFF
        else:
            val = (val * 2654435761 + int(k) + 1013904223) & 0xFFFFFFFF
    return np.random.default_rng(val)


# ============================================================
# Configuration
# ============================================================

mnist_dir = '/home/dataset/mnist'
output_root = '/home/dataset/cmnist_border_fast_slow_only'

image_size = 48
core_size = 42
border_width = (image_size - core_size) // 2
cell_pixel = core_size * core_size
m = 128
core_background_rgb_uint8 = (m, m, m)

cell_digit_fraction = 0.14
cell_digit_pixels = round(cell_digit_fraction * cell_pixel)
cell_bg_pixels = cell_pixel - cell_digit_pixels

condition_specs = [
    {
        'condition_index': 0,
        'condition_id': 'fast_r10',
        'condition_name': 'fast_red_plus_10pct',
        'p': 10,
        'channel': 'R',
        'channel_index': 0,
        'delta_fraction': 0.10,
    },
    {
        'condition_index': 1,
        'condition_id': 'slow_g45',
        'condition_name': 'slow_green_plus_45pct',
        'p': 45,
        'channel': 'G',
        'channel_index': 1,
        'delta_fraction': 0.45,
    },
]
condition_ids = [spec['condition_id'] for spec in condition_specs]
condition_by_id = {spec['condition_id']: spec for spec in condition_specs}

# Border calibration: almost fixed R/B, highly variable G.
# This makes empirical sigma_G much larger than sigma_R.
border_noise_specs = {
    'R': {'mode': 'uniform_int', 'low': -1, 'high': 1},
    'G': {'mode': 'uniform_int', 'low': -127, 'high': 127},
    'B': {'mode': 'uniform_int', 'low': -1, 'high': 1},
}

shuffle_source_pool_per_condition = True
min_mask_ratio = 0.12
bias = 0.05
seed = 0

print('C-MNIST border FAST/SLOW-only generator configuration')
print(f'  image size       : {image_size} x {image_size}')
print(f'  core size        : {core_size} x {core_size}')
print(f'  border width     : {border_width}')
print(f'  condition ids    : {condition_ids}')
print(f'  p values         : {[spec["p"] for spec in condition_specs]}')
print(f'  core mean gray   : {m}')
print(f'  digit pixels     : {cell_digit_pixels} / {cell_pixel} core pixels')
print(f'  border noise     : {border_noise_specs}')


# ============================================================
# Rendering functions
# ============================================================


def sample_border_color_uint8(seed, split_idx, sample_idx, condition_id):
    rng = deterministic_rng(seed, 'border', split_idx, sample_idx, condition_id)
    out = []
    for key in ['R', 'G', 'B']:
        spec = border_noise_specs[key]
        if spec['mode'] == 'constant':
            delta = int(spec['value'])
        elif spec['mode'] == 'uniform_int':
            delta = int(rng.integers(spec['low'], spec['high'] + 1))
        else:
            raise ValueError(spec['mode'])
        out.append(clip_uint8_value(m + delta))
    return tuple(out)


def condition_digit_rgb_uint8(spec):
    rgb = [int(m), int(m), int(m)]
    delta_uint8 = clip_uint8_value(float(spec['delta_fraction']) * 255.0)
    rgb[int(spec['channel_index'])] = clip_uint8_value(int(m) + delta_uint8)
    return tuple(rgb), int(delta_uint8)


def paint_fast_slow_cmnist(image, condition_id, border_rgb_uint8):
    spec = condition_by_id[condition_id]
    digit_mask = get_exact_digit_mask(image, cell_digit_pixels, image_size=core_size, bias=bias).bool()
    digit_rgb_uint8, _ = condition_digit_rgb_uint8(spec)

    out = torch.zeros(3, image_size, image_size, dtype=torch.float32)
    border_rgb = torch.as_tensor(rgb_uint8_to_float(border_rgb_uint8), dtype=torch.float32).view(3, 1, 1)
    core_bg_rgb = torch.as_tensor(rgb_uint8_to_float(core_background_rgb_uint8), dtype=torch.float32).view(3, 1, 1)
    digit_rgb = torch.as_tensor(rgb_uint8_to_float(digit_rgb_uint8), dtype=torch.float32)

    out[:] = border_rgb
    core = out[:, border_width:border_width + core_size, border_width:border_width + core_size]
    core[:] = core_bg_rgb
    core[:, digit_mask] = digit_rgb.view(3, 1)
    return out


# ============================================================
# Dataset generation
# ============================================================

source_records, filtered_img_ids = load_filtered_mnist(mnist_dir, min_mask_ratio=min_mask_ratio, bias=bias)

# Preview
set_seed(seed)
example_source = source_records[0][0]['image']
fig, axes = plt.subplots(1, len(condition_ids), figsize=(3.5 * len(condition_ids), 3.2))
if len(condition_ids) == 1:
    axes = [axes]
for ax, cid in zip(axes, condition_ids):
    border_rgb = sample_border_color_uint8(seed, 0, 0, cid)
    rendered = paint_fast_slow_cmnist(example_source, cid, border_rgb)
    digit_rgb, delta_uint8 = condition_digit_rgb_uint8(condition_by_id[cid])
    ax.imshow(tensor_to_pil_image(rendered))
    ax.axis('off')
    ax.set_title(f"{cid}\nraw Δ={delta_uint8}\nborder={border_rgb}", fontsize=8)
plt.tight_layout()
plt.show()

output_dir = os.path.join(output_root, f'fastslow_m{m}_seed{seed}')
os.makedirs(output_dir, exist_ok=True)

all_train_image_paths = []

for split_idx, split in enumerate(['train', 'test']):
    source_pool = source_records[split_idx]
    N = len(source_pool)
    source_pool_hist = label_histogram_from_records(source_pool)
    condition_permutations = make_condition_source_permutations(
        num_sources=N,
        condition_ids_order=condition_ids,
        seed_value=seed,
        split_idx=split_idx,
        shuffle_source_pool_per_condition=shuffle_source_pool_per_condition,
    )

    # Hard verification before generation.
    for cid in condition_ids:
        assert sorted(condition_permutations[cid]) == list(range(N))
        assert label_histogram_from_permutation(source_pool, condition_permutations[cid]) == source_pool_hist

    image_root = os.path.join(output_dir, split, 'images')
    for cid in condition_ids:
        os.makedirs(os.path.join(image_root, cid), exist_ok=True)

    manifest_path = os.path.join(output_dir, split, 'manifest.csv')
    manifest_long_path = os.path.join(output_dir, split, 'manifest_long.csv')

    rows = []
    long_rows = []

    for sample_idx in tqdm(range(N), desc=f'generate {split}'):
        sample_id = f'{sample_idx:06d}'
        row = {'sample_id': sample_id}

        for cond_idx, cid in enumerate(condition_ids):
            source_pool_pos = int(condition_permutations[cid][sample_idx])
            source_rec = source_pool[source_pool_pos]
            border_rgb_uint8 = sample_border_color_uint8(seed, split_idx, sample_idx, cid)
            rendered = paint_fast_slow_cmnist(source_rec['image'], cid, border_rgb_uint8)

            rel_path = f'images/{cid}/{sample_id}.png'
            abs_path = os.path.join(output_dir, split, rel_path)
            tensor_to_pil_image(rendered).save(abs_path)
            if split == 'train':
                all_train_image_paths.append(abs_path)

            spec = condition_by_id[cid]
            digit_rgb_uint8, delta_uint8 = condition_digit_rgb_uint8(spec)

            row[f'cond{cond_idx}_image'] = rel_path
            row[f'cond{cond_idx}_label'] = int(source_rec['label'])
            row[f'cond{cond_idx}_source_pool_index'] = int(source_rec['source_pool_index'])
            row[f'cond{cond_idx}_source_mnist_index'] = int(source_rec['source_mnist_index'])
            row[f'cond{cond_idx}_condition'] = cid
            row[f'cond{cond_idx}_p'] = int(spec['p'])
            row[f'cond{cond_idx}_channel'] = spec['channel']
            row[f'cond{cond_idx}_delta_fraction'] = float(spec['delta_fraction'])
            row[f'cond{cond_idx}_delta_uint8'] = int(delta_uint8)
            row[f'cond{cond_idx}_digit_rgb_uint8'] = json.dumps(list(digit_rgb_uint8))
            row[f'cond{cond_idx}_border_rgb_uint8'] = json.dumps(list(border_rgb_uint8))

            long_rows.append({
                'sample_id': sample_id,
                'image': rel_path,
                'label': int(source_rec['label']),
                'source_pool_index': int(source_rec['source_pool_index']),
                'source_mnist_index': int(source_rec['source_mnist_index']),
                'condition_index': int(spec['condition_index']),
                'condition': cid,
                'condition_name': spec['condition_name'],
                'p': int(spec['p']),
                'channel': spec['channel'],
                'channel_index': int(spec['channel_index']),
                'delta_fraction': float(spec['delta_fraction']),
                'delta_uint8': int(delta_uint8),
                'digit_rgb_uint8': json.dumps(list(digit_rgb_uint8)),
                'core_bg_rgb_uint8': json.dumps(list(core_background_rgb_uint8)),
                'border_rgb_uint8': json.dumps(list(border_rgb_uint8)),
            })
        rows.append(row)

    fieldnames = ['sample_id']
    for cond_idx in range(len(condition_ids)):
        fieldnames += [
            f'cond{cond_idx}_image',
            f'cond{cond_idx}_label',
            f'cond{cond_idx}_source_pool_index',
            f'cond{cond_idx}_source_mnist_index',
            f'cond{cond_idx}_condition',
            f'cond{cond_idx}_p',
            f'cond{cond_idx}_channel',
            f'cond{cond_idx}_delta_fraction',
            f'cond{cond_idx}_delta_uint8',
            f'cond{cond_idx}_digit_rgb_uint8',
            f'cond{cond_idx}_border_rgb_uint8',
        ]
    with open(manifest_path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

    long_fieldnames = list(long_rows[0].keys()) if long_rows else []
    with open(manifest_long_path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=long_fieldnames)
        writer.writeheader()
        writer.writerows(long_rows)

    condition_label_hist = {
        cid: [sum((r['condition'] == cid and int(r['label']) == label) for r in long_rows) for label in range(10)]
        for cid in condition_ids
    }
    assert all(condition_label_hist[cid] == source_pool_hist for cid in condition_ids)

    print(f'{split}: wrote {len(long_rows)} PNG files for {N} source MNIST images x {len(condition_ids)} conditions')
    print(f'  manifest      : {manifest_path}')
    print(f'  manifest_long : {manifest_long_path}')
    print(f'  source label histogram: {source_pool_hist}')
    print(f'  condition label histograms identical: {all(condition_label_hist[cid] == source_pool_hist for cid in condition_ids)}')


# ============================================================
# Save empirical normalization metadata
# ============================================================

train_mean, train_std = compute_empirical_stats(all_train_image_paths)
print('empirical train mean:', train_mean)
print('empirical train std :', train_std)

condition_meta = {}
for cid in condition_ids:
    spec = condition_by_id[cid]
    digit_rgb_uint8, delta_uint8 = condition_digit_rgb_uint8(spec)
    channel_index = int(spec['channel_index'])
    raw_delta_float = float(delta_uint8) / 255.0
    effective_amp = raw_delta_float / max(float(train_std[channel_index]), 1e-12)
    condition_meta[cid] = {
        **spec,
        'delta_uint8': int(delta_uint8),
        'digit_rgb_uint8': list(digit_rgb_uint8),
        'digit_hex': rgb_hex_from_uint8(digit_rgb_uint8),
        'core_bg_rgb_uint8': list(core_background_rgb_uint8),
        'core_bg_hex': rgb_hex_from_uint8(core_background_rgb_uint8),
        'raw_delta_float': raw_delta_float,
        'empirical_channel_std': float(train_std[channel_index]),
        'normalized_effective_amp': float(effective_amp),
        'normalized_effective_energy': float(effective_amp ** 2),
    }

colors_metadata = {
    # Top-level m kept for compatibility with older loaders.
    'm': int(m),
    # New loaders should use this std for empirical normalization.
    'std': [float(x) for x in train_std],
    'mean_rgb_empirical': [float(x) for x in train_mean],
    'config': {
        'design': 'cmnist_border_fast_slow_only',
        'description': 'Two-condition fast/slow C-MNIST. A label-independent border inflates green-channel empirical std so a raw-larger green digit feature can become normalized-smaller than a raw-smaller red digit feature.',
        'image_size': int(image_size),
        'core_size': int(core_size),
        'border_width': int(border_width),
        'core_mean_gray_uint8': int(m),
        'border_noise_specs': border_noise_specs,
        'condition_ids': condition_ids,
        'condition_specs': condition_specs,
        'seed': int(seed),
        'training_note': 'Use p_values=[10,45]. If your loader hard-codes NUM_CONDITIONS=4, set it to 2 or infer cond columns.',
    },
    'conditions': condition_meta,
}
with open(os.path.join(output_dir, 'colors.json'), 'w') as f:
    json.dump(colors_metadata, f, indent=2)
with open(os.path.join(output_dir, 'mean.json'), 'w') as f:
    json.dump({'m': int(m), 'empirical_mean_rgb': [float(x) for x in train_mean]}, f, indent=2)
with open(os.path.join(output_dir, 'stats.json'), 'w') as f:
    json.dump({'train_mean_rgb': [float(x) for x in train_mean], 'train_std_rgb': [float(x) for x in train_std]}, f, indent=2)

print('wrote metadata to:', output_dir)
print('\n[Effective amplitudes after empirical normalization]')
for cid in condition_ids:
    meta = condition_meta[cid]
    print(
        f"  {cid:>9s}: raw_delta={meta['delta_uint8']:>3d}, "
        f"channel={meta['channel']}, std={meta['empirical_channel_std']:.6f}, "
        f"eff_amp={meta['normalized_effective_amp']:.4f}, "
        f"eff_energy={meta['normalized_effective_energy']:.4f}"
    )

if 'fast_r10' in condition_meta and 'slow_g45' in condition_meta:
    fast = condition_meta['fast_r10']
    slow = condition_meta['slow_g45']
    ratio = fast['normalized_effective_amp'] / max(slow['normalized_effective_amp'], 1e-12)
    print(f"\nfast_r10 / slow_g45 normalized amplitude ratio: {ratio:.4f}")
    print('order flip success:', ratio > 1.0)


In [ ]:

# %%
# Optional sanity check cell. Run after generation.
import os, csv, json
from collections import defaultdict, Counter
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt


def _parse_json_list(x):
    if isinstance(x, (list, tuple)):
        return list(x)
    return json.loads(x)


def _load_long_manifest(split="train"):
    path = os.path.join(output_dir, split, "manifest_long.csv")
    rows = []
    with open(path, "r", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            rows.append(row)
    return rows


def _compute_empirical_stats_from_manifest(split="train"):
    rows = _load_long_manifest(split)
    s = np.zeros(3, dtype=np.float64)
    ss = np.zeros(3, dtype=np.float64)
    n = 0
    for row in rows:
        img_path = os.path.join(output_dir, split, row["image"])
        arr = np.asarray(Image.open(img_path).convert("RGB"), dtype=np.float64) / 255.0
        flat = arr.reshape(-1, 3)
        s += flat.sum(axis=0)
        ss += (flat ** 2).sum(axis=0)
        n += flat.shape[0]
    mean = s / n
    var = np.maximum(ss / n - mean ** 2, 0.0)
    std = np.sqrt(var)
    return mean, std, rows

train_mean2, train_std2, train_rows = _compute_empirical_stats_from_manifest("train")
print("output_dir:", output_dir)
print("mean:", train_mean2)
print("std :", train_std2)

with open(os.path.join(output_dir, "stats.json"), "r") as f:
    saved_stats = json.load(f)
print("saved std close:", np.allclose(train_std2, np.asarray(saved_stats["train_std_rgb"]), atol=1e-8))

# Label histogram check
label_hist_by_condition = defaultdict(Counter)
for row in train_rows:
    label_hist_by_condition[row["condition"]][int(row["label"])] += 1
print("\nlabel histograms")
for cid in condition_ids:
    hist = [label_hist_by_condition[cid][i] for i in range(10)]
    print(cid, hist)
print("identical:", len({tuple(label_hist_by_condition[cid][i] for i in range(10)) for cid in condition_ids}) == 1)

records = []
for cid in condition_ids:
    spec = condition_by_id[cid]
    channel_index = int(spec["channel_index"])
    raw_delta_uint8 = int(round(float(spec["delta_fraction"]) * 255.0))
    raw_delta_float = raw_delta_uint8 / 255.0
    effective_amp = raw_delta_float / max(float(train_std2[channel_index]), 1e-12)
    records.append({
        "condition": cid,
        "p": int(spec["p"]),
        "channel": spec["channel"],
        "raw_delta_uint8": raw_delta_uint8,
        "raw_delta_float": raw_delta_float,
        "empirical_channel_std": float(train_std2[channel_index]),
        "normalized_effective_amp": effective_amp,
        "normalized_effective_energy": effective_amp ** 2,
    })
amp_df = pd.DataFrame(records).sort_values("normalized_effective_energy", ascending=False)
display(amp_df)
print("expected order:", " > ".join(amp_df["condition"].tolist()))

amp_by_cond = {r["condition"]: r for r in records}
if "fast_r10" in amp_by_cond and "slow_g45" in amp_by_cond:
    fast = amp_by_cond["fast_r10"]
    slow = amp_by_cond["slow_g45"]
    print("\n[fast_r10 vs slow_g45]")
    print("raw amp ratio fast/slow:", fast["raw_delta_float"] / slow["raw_delta_float"])
    print("norm amp ratio fast/slow:", fast["normalized_effective_amp"] / slow["normalized_effective_amp"])
    print("order flip success:", fast["normalized_effective_amp"] > slow["normalized_effective_amp"])

# Border independence diagnostic
border_rows = []
for row in train_rows:
    border_rgb = _parse_json_list(row["border_rgb_uint8"])
    border_rows.append({
        "condition": row["condition"],
        "label": int(row["label"]),
        "border_R": int(border_rgb[0]),
        "border_G": int(border_rgb[1]),
        "border_B": int(border_rgb[2]),
    })
border_df = pd.DataFrame(border_rows)
print("\nBorder RGB mean/std by condition")
display(border_df.groupby("condition")[["border_R", "border_G", "border_B"]].agg(["mean", "std"]))
print("\nBorder-label correlations")
for ch in ["border_R", "border_G", "border_B"]:
    corr = np.corrcoef(border_df[ch].to_numpy(dtype=np.float64), border_df["label"].to_numpy(dtype=np.float64))[0, 1]
    print(f"{ch}: {corr:.6f}")

fig, axes = plt.subplots(1, len(condition_ids), figsize=(3.5 * len(condition_ids), 3.2))
if len(condition_ids) == 1:
    axes = [axes]
example_rows_by_cond = {}
for row in train_rows:
    if row["condition"] not in example_rows_by_cond:
        example_rows_by_cond[row["condition"]] = row
for ax, cid in zip(axes, condition_ids):
    row = example_rows_by_cond[cid]
    img_path = os.path.join(output_dir, "train", row["image"])
    ax.imshow(Image.open(img_path).convert("RGB"))
    ax.axis("off")
    rec = amp_by_cond[cid]
    ax.set_title(f"{cid}\nraw Δ={rec['raw_delta_uint8']}\neff={rec['normalized_effective_amp']:.2f}", fontsize=9)
plt.tight_layout()
plt.show()
